# 03 — Multi-Head Attention — Solution

---

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.abspath('../..'))
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Tuple
from src.models.attention import scaled_dot_product_attention, MultiHeadAttention, GroupedQueryAttention

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Part B — MultiHeadAttention Implementation

In [ ]:
class MyMultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.dropout = dropout

        # Four weight matrices, no bias (common in modern LLMs)
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, query, key, value, mask=None, return_attention=False):
        batch = query.size(0)

        # Step 1 — project + split into heads: (batch, n_heads, seq, d_k)
        q = self.w_q(query).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)
        k = self.w_k(key).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)
        v = self.w_v(value).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)

        # Step 2 — attend across all heads simultaneously
        output, attn_weights = scaled_dot_product_attention(
            q, k, v, mask=mask, dropout=self.dropout, training=self.training
        )

        # Step 3 — merge heads: (batch, seq, d_model)
        output = output.transpose(1, 2).contiguous().view(batch, -1, self.d_model)
        output = self.w_o(output)

        if return_attention:
            return output, attn_weights
        return output, None


mha = MyMultiHeadAttention(d_model=128, n_heads=4)
x = torch.randn(2, 10, 128)
out, w = mha(x, x, x, return_attention=True)
print(f'Output : {out.shape}')    # (2, 10, 128)
print(f'Weights: {w.shape}')     # (2, 4, 10, 10)

## Part B2 — Parameter Count

In [ ]:
d_model = 512
# W_q, W_k, W_v, W_o  each (d_model × d_model), no bias
expected_params = 4 * d_model * d_model
mha_512 = MyMultiHeadAttention(d_model=512, n_heads=8)
actual = sum(p.numel() for p in mha_512.parameters())
print(f'Expected : {expected_params:,}')
print(f'Actual   : {actual:,}')
print(f'Match    : {expected_params == actual}')

## Part C — Head Visualisation

In [ ]:
sentence = "the quick brown fox jumps over the lazy dog"
words = sentence.split()
vocab = {w: i for i, w in enumerate(sorted(set(words)))}
d_model, n_heads = 64, 4
embed = nn.Embedding(len(vocab), d_model)
tokens = torch.tensor([vocab[w] for w in words])
x = embed(tokens).unsqueeze(0)

mha_vis = MultiHeadAttention(d_model=d_model, n_heads=n_heads)
_, weights = mha_vis(x, x, x, return_attention=True)

fig, axes = plt.subplots(1, n_heads, figsize=(4 * n_heads, 4))
for h, ax in enumerate(axes):
    w_np = weights[0, h].detach().numpy()
    im = ax.imshow(w_np, cmap='Blues', vmin=0)
    ax.set_title(f'Head {h+1}')
    ax.set_xticks(range(len(words))); ax.set_xticklabels(words, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(len(words))); ax.set_yticklabels(words, fontsize=7)
    plt.colorbar(im, ax=ax)
plt.suptitle('Multi-Head Attention — each head learns a different pattern', fontsize=11)
plt.tight_layout(); plt.show()

## Part D — GQA Memory Analysis

In [ ]:
gqa = GroupedQueryAttention(d_model=128, n_heads=8, n_kv_heads=2)
x = torch.randn(2, 10, 128)
out, _ = gqa(x, x, x)
print(f'GQA output: {out.shape}')

def kv_cache_size_mb(n_layers, seq_len, n_kv_heads, d_k, bytes_per_elem=2):
    return 2 * n_layers * seq_len * n_kv_heads * d_k * bytes_per_elem / 1e6

n_layers, seq_len, d_model, n_heads = 28, 2048, 3072, 16
d_k = d_model // n_heads
mha_mb  = kv_cache_size_mb(n_layers, seq_len, n_heads, d_k)
gqa2_mb = kv_cache_size_mb(n_layers, seq_len, 2, d_k)
gqa1_mb = kv_cache_size_mb(n_layers, seq_len, 1, d_k)

print(f'\nKV-cache at seq_len={seq_len}:')
print(f'  MHA  ({n_heads} KV heads): {mha_mb:.1f} MB')
print(f'  GQA  (2 KV heads ): {gqa2_mb:.1f} MB   ({mha_mb/gqa2_mb:.0f}x reduction)')
print(f'  MQA  (1 KV head  ): {gqa1_mb:.1f} MB   ({mha_mb/gqa1_mb:.0f}x reduction)')

## Summary

**Key insight:** Multi-head attention multiplies the representational capacity at essentially no extra FLOP cost relative to a single head with the same total dimension. GQA/MQA cut KV-cache memory by 8–16x, enabling much longer context windows at inference.

**Next:** `04_positional_encoding_starter.ipynb`